# **Not all samples are created equal : Deep Learning with Importance Sampling**

Angelos Katharopoulos, François Fleuret (2019)

Aujourd'hui, des méthodes de Deep Learning très évoluées et un accès à la data démocratisé permettent d'entraîner des réseaux de neurones de plus en plus complexes. Dans ce cadre-ci, il devient important de prêter attention au temps d'entraînement de tels modèles, qui peut vite devenir considérable.  

Or, au sein d'un ensemble d'entraînement, un grand nombre d'échantillons sont vite apprivoisés par le modèle, qui les classent correctement au bout de quelques epochs seulement. Ils ne lui apprennent ensuite plus rien, mais l'on continue de les faire passer à travers le réseau à chaque itération, ce qui prend un temps colossal. Il convient donc de vouloir échantillonner nos données en choisissant simplement celles qui vont faire progresser le modèle le plus vite possible, afin d'utiliser bien moins d'échantillons, pour un résultat tout aussi efficace.  

C'est le principe de l'Importance Sampling, décrit dans l'article en question datant de 2019. L'objectif ici sera de trouver une manière intelligente de choisir les échantillons sur lesquels le modèle s'appuiera, en se basant sur le gradient de leur loss (l'écart entre leur valeur prédite et leur valeur réelle).

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import time

class ImportanceSamplingTrainer:
    def __init__(self, model, optimizer, B=256, b=32, tau_th=1.2, a_tau=0.9):
        """
        B : Taille du réservoir (le grand échantillon U)
        b : Taille du batch final (le petit échantillon G)
        tau_th : Seuil de rentabilité pour activer l'Importance Sampling
        a_tau : Coefficient de lissage pour la moyenne mobile de tau
        """
        self.model = model
        self.optimizer = optimizer
        self.B = B
        self.b = b
        self.tau_th = tau_th
        self.a_tau = a_tau
        self.tau = 0.0  # Initialisé à 0 pour commencer en Uniforme
        self.criterion_none = nn.CrossEntropyLoss(reduction='none')
        self.device = next(model.parameters()).device

    def compute_tau(self, gi):
        """ Équation 27 : Estimation du gain de réduction de variance """
        numerator = np.sum(gi**2)
        mean_g = np.mean(gi)
        denominator = np.sum((gi - mean_g)**2)

        if denominator == 0: return 1.0
        # On calcule le tau instantané
        tau_t = 1.0 / (1.0 - (denominator / numerator))
        return tau_t

    def train_one_step(self, dataloader_iter):
        # 1. Décision : IS ou Uniforme ? (Ligne 5)
        use_is = self.tau > self.tau_th

        if use_is:
            # --- MODE IMPORTANCE SAMPLING ---
            # On récupère B exemples (Réservoir U)
            images_U, labels_U = next(dataloader_iter)
            images_U, labels_U = images_U.to(self.device), labels_U.to(self.device)

            # Calcul de G_hat (Ligne 7) : Forward pass rapide
            self.model.eval()
            with torch.no_grad():
                outputs_U = self.model(images_U)
                # Approximation : Erreur L2 entre probas et labels (Equation 20 simplifiée)
                probs_U = torch.softmax(outputs_U, dim=1)
                one_hot_labels = torch.nn.functional.one_hot(labels_U, num_classes=outputs_U.size(1))
                # gi = borne supérieure du gradient (norme de l'erreur en sortie)
                gi = torch.norm(probs_U - one_hot_labels, p=2, dim=1).cpu().numpy()

            # Sélection de b exemples (Ligne 8)
            p_sampling = gi / gi.sum()
            indices = np.random.choice(self.B, size=self.b, p=p_sampling, replace=False)

            images_G = images_U[indices]
            labels_G = labels_U[indices]

            # Calcul des poids wi (Ligne 9) : Pour rester non-biaisé
            wi = 1.0 / (self.B * p_sampling[indices])
            wi = torch.from_numpy(wi).to(self.device).float()
            # Normalisation des poids pour la stabilité numérique
            wi = wi / wi.mean()

        else:
            # --- MODE UNIFORME ---
            # On prend juste b exemples au hasard (Ligne 12)
            images_G, labels_G = next(dataloader_iter)
            images_G, labels_G = images_G.to(self.device), labels_G.to(self.device)
            wi = torch.ones(self.b).to(self.device)

            # On calcule quand même gi pour mettre à jour tau (Ligne 15)
            self.model.eval()
            with torch.no_grad():
                outputs_G = self.model(images_G)
                probs_G = torch.softmax(outputs_G, dim=1)
                one_hot_labels = torch.nn.functional.one_hot(labels_G, num_classes=outputs_G.size(1))
                gi = torch.norm(probs_G - one_hot_labels, p=2, dim=1).cpu().numpy()

        # 2. Mise à jour des poids (Backprop sur b échantillons)
        self.model.train()
        self.optimizer.zero_grad()
        outputs = self.model(images_G)
        # On applique les poids wi sur la loss de chaque échantillon
        loss = (self.criterion_none(outputs, labels_G) * wi).mean()
        loss.backward()
        self.optimizer.step()

        # 3. Mise à jour de tau (Ligne 17) : Moyenne mobile
        tau_t = self.compute_tau(gi)
        self.tau = self.a_tau * self.tau + (1 - self.a_tau) * tau_t

        return loss.item(), self.tau